# Criterium Research API Tutorial
This notebook walks through the Gemini-powered research API.

**Prerequisites:**
1. Ensure your server is running (`make run` in a terminal).
2. Ensure your `.env` file contains a valid Gemini API key. A Firecrawl key is only needed when using the alternative Firecrawl researcher.

In [ ]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

def print_json(data):
    # default=str handles non-JSON-native types like datetime
    print(json.dumps(data, indent=2, default=str))

## 1. Create a Research Collection
Since we are using an in-memory SQLite database (`:memory:`), each server restart begins with an empty database. Let's create our test collection first using the Book Series schema.

Notice that because we share the `schemas` between the client and server, we can build the schema using pure Pydantic and never write raw JSON manually!

In [ ]:
from criterium import schemas

schema = schemas.ResearchSchemaObject(
  properties={
    "title": schemas.ResearchSchemaString(
      description="The exact title of the primary book.",
      is_required=True
    ),
    "year": schemas.ResearchSchemaInteger(
      description="The 4-digit year the book was published.",
      is_required=True
    ),
    "series_books": schemas.ResearchSchemaArray(
      description="Other books in the same series.",
      items=schemas.ResearchSchemaObject(
        properties={
          "title": schemas.ResearchSchemaString(is_required=True),
          "chronological_order": schemas.ResearchSchemaInteger()
        }
      )
    )
  }
)

# Use our strictly typed Pydantic models to create the payload!
payload = schemas.CollectionCreate(
    name="Book Series Researcher",
    extraction_prompt="You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.",
    research_schema=schema
)

try:
    response = requests.post(f"{BASE_URL}/collections", json=payload.model_dump(exclude_none=True))
    response.raise_for_status()
    
    # We can validate the response straight back into our Pydantic model
    collection_data = schemas.CollectionResponse.model_validate(response.json())
    collection_id = collection_data.id
    print(f"Created collection with ID: {collection_id}")
    print_json(collection_data.model_dump())
    
except requests.exceptions.ConnectionError:
    print("Error: Could not connect to the server at", BASE_URL)
    print("Please make sure your server is running in a terminal with: make run")
except Exception as e:
    print("Request failed:", e)

## 2. Verify / List All Collections
Now let's query `GET /collections` to verify our newly created collection is listed in the database.

In [ ]:
response = requests.get(f"{BASE_URL}/collections")

# Parse list of Pydantic models
collections = [schemas.CollectionResponse.model_validate(c) for c in response.json()]

print(f"Found {len(collections)} collections.")
for col in collections:
    print(f"- {col.id}: {col.name}")

## 3. Research a Product
Now we'll trigger the Gemini 3.7 Flash research pipeline with Google Search grounding.
Behind the scenes, this will:
1. Browse the web for evidence relevant to the collection prompt and schema
2. Return schema-constrained extracted data and an evidence summary
3. Capture Gemini's cited web references
4. Save the result and references to SQLite in one transaction

In [ ]:
# This requires a valid GEMINI_API_KEY in the root .env file
query_payload = schemas.ProductCreate(
    product_info="The Lord of the Rings: The Fellowship of the Ring book"
)

print(f"Triggering grounded research for collection {collection_id}...")
extract_response = requests.post(
    f"{BASE_URL}/collections/{collection_id}/products",
    json=query_payload.model_dump(),
)

print("Status Code:", extract_response.status_code)
print_json(extract_response.json())

## 4. View the Extracted Data
Finally, let's pull down the products we just researched to see the raw markdown and the structured JSON output!

In [ ]:
products_response = requests.get(f"{BASE_URL}/collections/{collection_id}/products")
products_response.raise_for_status()

products = [schemas.ProductResponse.model_validate(p) for p in products_response.json()]

for product in products:
    print("=" * 50)
    print("PRODUCT:", product.name)
    print("PRIMARY SOURCE:", product.source_url)
    print("-" * 50)
    print("EXTRACTED DATA:")
    print_json(product.extracted_data)
    print("REFERENCES:")
    for reference in product.references:
        print(f"- {reference.title or reference.url} ({reference.provider})")
        print(f"  {reference.url}")
    print("=" * 50)